In [70]:
import random
from qdrant_client import QdrantClient

import instructor
import json
from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken

In [71]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [72]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Load all data from Qdrant

In [73]:
all_points = []
next_offset = None
batch_size = 100

while True:
    points, next_offset = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        limit=batch_size,
        offset=next_offset,
        with_payload=True,
        with_vectors=False
    )
    all_points.extend(points)
    if next_offset is None:
        break

In [74]:
len(all_points)

1000

### Split the data into 2 groups: 50 items for synthetic question generation, 950 items to loop through and evaluate their relevance for previously generated synthetic questions.

In [75]:
rng = random.Random(42)
indices = list(range(len(all_points)))

In [76]:
rng.shuffle(indices)

In [77]:
sampl_idx = set(indices[:50])

In [78]:
sample_50 = [p for i, p in enumerate(all_points) if i in sampl_idx]
remaining_points = [p for i, p in enumerate(all_points) if i not in sampl_idx]

In [79]:
len(sample_50), len(remaining_points)

(50, 950)

### Generate synhetic questions
- pass all the information to llm, llm will return a list of synthetic questions; model is less likely to hallucinate on the id

In [80]:
all_context_sample_50 = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in sample_50]

In [81]:
all_context_sample_50

[{'id': 'B07PHXDR1R',
  'text': '128GB USB Flash Drive,DISAIN Dual Drive USB 3.0 Type-C 3.1 Gen 1 Devices, Thumb Drive Memory Stick for Computers ( Red) Flash drive for USB Type-C devices, with a reversible USB Type-C connector and a traditional USB connector. Reliable and secure storage for your videos, music, photos, or other files. Easily, conveniently transfer files between smartphones, tablets and computers, that are with type c charging port. Free up space on Android phone (Device requires USB Type-C port and On-The-Go (OTG) support). High speed USB 3.0 performance of up to 85MB/s Read Speeds and 40MB/s Write Speeds. Most mobile phones, tablets and personal computers with type c charging ports are currently supported. The DIP material double-fixed patch Type-C head has a firm welding line, and the head has a longer extension, made of stainless steel and does not rust long time. The USB 3.0 head is made of premium materials, has a beautiful and bright appearance and stable perform

In [82]:
all_context_remaining_points = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in remaining_points]

In [83]:
len(all_context_remaining_points)

950

In [84]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that a large number of products could satisfy.

## Products
{json.dumps(all_context_sample_50, indent=2)}
"""

In [85]:
print(SYSTEM_PROMPT)


You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
-

In [86]:
client = instructor.from_provider(
    "openai/gpt-5.4", # be same, as it is a hard task
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [87]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

In [88]:
# create class that will hold array in output schema
from typing import Annotated


class SyntheticQuestion(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question: str = Field(description="Suggested question.")
    chunk_ids: list[str] = Field(description="ID of the chunk that could be used to answer the question.")
    answer_example: str = Field(description="Suggested answer grounded in the context.")


class SyntheticQuestions(BaseModel):
    synthetic_questions: Annotated[
        list[SyntheticQuestion],
        Field(min_length=30, max_length=30, description="Exactly 30 synthetic questions."),
    ]


In [89]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestions
)

In [90]:
len(response.synthetic_questions)

30

In [91]:
response.synthetic_questions

[SyntheticQuestion(reasoning='This asks about a dual USB-C and USB-A flash drive with a stated capacity and transfer use, which is fully covered by the DISAIN flash drive product.', question='Do you have a 128GB flash drive that works with both USB-C phones and regular USB ports on my computer?', chunk_ids=['B07PHXDR1R'], answer_example='Yes, there is a 128GB DISAIN dual drive flash drive with both a USB Type-C connector and a traditional USB connector for transferring files between phones, tablets, and computers.'),
 SyntheticQuestion(reasoning='This asks for an outdoor TV antenna with long range and support for two TVs, which matches the ViewTV antenna description exactly.', question="I'm looking for an outdoor TV antenna that can reach about 150 miles and connect to two TVs at once. Do you have one?", chunk_ids=['B01CYX4OZ2'], answer_example='Yes. The ViewTV outdoor amplified digital HD antenna supports up to a 150-mile range and can connect to 2 televisions at the same time without

In [92]:
from typing import Any

from collections import defaultdict
check_response = defaultdict[Any, int](int)
for question in response.synthetic_questions:
    check_response[len(question.chunk_ids)] += 1

check_response.items(), len(response.synthetic_questions)    

(dict_items([(1, 15), (2, 10), (0, 5)]), 30)

In [93]:
# see if other context can be used to answer the question
synthetic_questions_relevant_data = [{"question_id": i, "question": item.question} 
            for i, item in enumerate(response.synthetic_questions) ]


### assign renaming chunks to relevant questions

In [94]:
SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
{json.dumps(all_context_remaining_points[0:50], indent=2)}
"""

In [95]:
len(synthetic_questions_relevant_data)

30

In [96]:
print(SYSTEM_PROMPT_950)


You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
[
  {
    "question_id": 0,
    "question": "Do you have a 128GB flash drive that works with both USB-C phones and regular USB ports on my computer?"
  },
  {
    

### Count the number of tokens for the static prefix
- prompt caching is only possible if 

In [97]:
SYSTEM_PROMPT_STATIC_PART = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
"""

In [98]:
def count_tokens(text):

    encoding = tiktoken.get_encoding("o200k_base")

    return len(encoding.encode(text))

In [99]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART))

1332


### Assign chunks to relevant questions for a single batch

In [100]:
client = instructor.from_provider(
    "openai/gpt-5.4-mini",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [101]:
class SyntheticQuestionMatchingChunk(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question_id: int = Field(description="ID of the question.")
    chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class SyntheticQuestionMatchingChunks(BaseModel):
    synthetic_question_matching_chunks: list[SyntheticQuestionMatchingChunk] = Field(description="List of synthetic questions and their matching chunks")

In [102]:
tmp_response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_950}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestionMatchingChunks
)

- do not want too many items can answer too many questions, so we want to take a look at the distribution

In [103]:
for data in tmp_response.synthetic_question_matching_chunks:
    print(f"Question ID: {data.question_id}, Chunk IDs: {data.chunk_ids}")

Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: []
Question ID: 2, Chunk IDs: []
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: []
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: []
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: []
Question ID: 19, Chunk IDs: []
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: []
Question ID: 25, Chunk IDs: ['B08PBQX47V']
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: ['B07J31M7PW']
Question ID: 29, Chunk IDs: []


In [104]:
raw_response.usage

ResponseUsage(input_tokens=13623, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=1804, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=15427)

In [105]:
def get_relevant_chunks(synthetic_questions, remaining_points_batch):

    SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
{json.dumps(remaining_points_batch, indent=2)}
"""

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_950}
        ],
        reasoning={"effort": "none"},
        response_model=SyntheticQuestionMatchingChunks
    )

    return response.synthetic_question_matching_chunks

In [106]:
answer_extended_chunks = []

for i in range(0, len(all_context_remaining_points), 50):

    print(f"Working on batch {(i+50)/50}")

    answer_extended_chunks.append(get_relevant_chunks(synthetic_questions_relevant_data, all_context_remaining_points[i:i+50]))

Working on batch 1.0
Working on batch 2.0
Working on batch 3.0
Working on batch 4.0
Working on batch 5.0
Working on batch 6.0
Working on batch 7.0
Working on batch 8.0
Working on batch 9.0
Working on batch 10.0
Working on batch 11.0
Working on batch 12.0
Working on batch 13.0
Working on batch 14.0
Working on batch 15.0
Working on batch 16.0
Working on batch 17.0
Working on batch 18.0
Working on batch 19.0


In [107]:
answer_extended_chunks

[[SyntheticQuestionMatchingChunk(reasoning='The question asks for a USB-C flash drive with 128GB capacity and compatibility with USB-C phones and regular USB ports. None of the products shown is a USB-C flash drive; the closest storage items are a generic USB data cable and a microSD card, but neither is sufficient alone.', question_id=0, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='The question asks for an outdoor TV antenna with about 150-mile range and support for two TVs. No product is an outdoor TV antenna at all.', question_id=1, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='The question asks for a USB-C to RCA cable for connecting a tablet to a stereo. The listed RCA product is an RCA-to-RCA cable, not USB-C to RCA, so it cannot independently answer the question.', question_id=2, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='The question asks for a USB-C to 3.5mm headphone adapter with calling support. No product is a USB-C to 3.5mm adapte

In [108]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        print(f"Question ID: {item.question_id}, Chunk IDs: {item.chunk_ids}")

Batch 1
Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: []
Question ID: 2, Chunk IDs: []
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: []
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: []
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: []
Question ID: 19, Chunk IDs: []
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: []
Question ID: 25, Chunk IDs: ['B08PBQX47V']
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: []
Question ID: 29, Chunk IDs: []
Batch 2
Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk 

In [109]:
valid_parent_asins = {point.payload["parent_asin"] for point in all_points}
questions_container = {}

for i, question in enumerate(response.synthetic_questions):
    questions_container[i] = []

    for chunk_id in question.chunk_ids:
        if chunk_id not in valid_parent_asins:
            print(f"Skipping hallucinated original chunk_id: {chunk_id}")
            continue

        if chunk_id not in questions_container[i]:
            questions_container[i].append(chunk_id)

In [110]:
valid_parent_asins = {point.payload["parent_asin"] for point in all_points}

for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        print(f"Question ID: {item.question_id}, Chunk IDs: {item.chunk_ids}")

        if item.question_id not in questions_container:
            print(f"Skipping unknown question_id: {item.question_id}")
            continue

        for chunk_id in item.chunk_ids:
            if chunk_id not in valid_parent_asins:
                print(f"Skipping hallucinated chunk_id: {chunk_id}")
                continue

            if chunk_id not in questions_container[item.question_id]:
                questions_container[item.question_id].append(chunk_id)

Batch 1
Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: []
Question ID: 2, Chunk IDs: []
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: []
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: []
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: []
Question ID: 19, Chunk IDs: []
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: []
Question ID: 25, Chunk IDs: ['B08PBQX47V']
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: []
Question ID: 29, Chunk IDs: []
Batch 2
Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk 

In [111]:
def get_description(parent_asin: str) -> str | None:
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=1,
    )[0]

    if not points:
        return None

    return points[0].payload["preprocessed_description"]

In [112]:
get_description("B09PFSVK44") # this is hallucination so not there

### generate goldend answer using llm for each of the question

In [113]:
def generate_golden_answer(question, relevant_chunks):

    class GoldenAnswer(BaseModel):
        answer: str = Field(description="The ideal reference answer for the question.")

    PROMPT = f"""
You are the shopping assistant of an e-shop. You are also being used to generate the ideal reference answer for a single customer question, to serve as ground truth for evaluating a RAG system.

## Context
A customer asked a question. A retrieval system has already gathered the products most relevant to it. Your job is to write the best possible answer a helpful shopping assistant could give, using ONLY those products.

## Inputs
- `question`: the customer's question.
- `products`: a list of relevant products, each with a `text` description. This may be empty or may not actually contain the answer.

## How to write the answer
- Ground every factual claim strictly in the provided product descriptions. Never invent products, prices, specs, availability, or details that are not present in the text.
- Answer in the voice of a friendly, concise shopping assistant speaking directly to the customer.
- Use only the information needed to answer well; don't dump every product if only some are relevant.
- If the products only partially answer the question, answer what you can and clearly state what information isn't available.
- If the products do not contain the information needed to answer at all (or the list is empty), do not guess. Respond the way a good assistant would when the shop doesn't carry the item or the info isn't available (e.g. politely say you couldn't find it), and do not fabricate an answer.

## Question
{json.dumps(question, indent=2)}

## Products
{json.dumps(relevant_chunks, indent=2)}"""

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": PROMPT}
        ],
        reasoning={"effort": "none"},
        response_model=GoldenAnswer
    )

    return response.answer

In [114]:
questions_container

{0: ['B07PHXDR1R', 'B08P1GD671', 'B07NLYRTK8', 'B017TJN22C'],
 1: ['B01CYX4OZ2', 'B083Q7Y3SJ'],
 2: ['B07TBD8J4L', 'B00UADFRJQ'],
 3: ['B096ZRXPKC', 'B07BGX6542'],
 4: ['B085T299YM', 'B08S7B31C3', 'B0828FLV9T', 'B095RHCV1T', 'B09J21NGVC'],
 5: ['B09DK2JH88'],
 6: ['B084ZLBS1Y', 'B09QYCF27R', 'B079FLLP6J'],
 7: ['B0829D5423'],
 8: ['B01MZ956T1'],
 9: ['B07L8BQB15', 'B00GGQRWQC'],
 10: ['B07VWS6BZ3'],
 11: ['B00B2IZ3B0', 'B078NNFHQ7', 'B088PKDWDJ'],
 12: ['B07VJMQP96', 'B09XKDYTYR'],
 13: ['B09YYDMFM1', 'B07GZL1V9B'],
 14: ['B07SJQNHTF', 'B08J42GSW6', 'B08FNNZTH1'],
 15: ['B01CYX4OZ2', 'B07RPCD3DL'],
 16: ['B07TBD8J4L',
  'B096ZRXPKC',
  'B0C54BYRTF',
  'B00UADFRJQ',
  'B0872NX4SJ',
  'B07GTKH3ZK',
  'B08D3G8F3J',
  'B0969XC1KL',
  'B07BGX6542'],
 17: ['B08DK692G2',
  'B0899GWMKB',
  'B07Z8S8JTH',
  'B07SDBZJBC',
  'B08DRCYKCR',
  'B083FV7KDS'],
 18: ['B01CVHARWU',
  'B0182BYHX4',
  'B00WIUIVJM',
  'B0085CFFFA',
  'B017U1248M',
  'B075992PHF',
  'B00FFYMJGE',
  'B072NGJG38'],
 19: ['B084

In [115]:
reference_dataset = []

for i, data in enumerate(response.synthetic_questions):
    print(f"Working on question {i+1}")

    reference_context_ids = []
    relevant_chunks = []

    for chunk_id in questions_container[i]:
        description = get_description(chunk_id)
        if description is None:
            print(f"Skipping missing description for chunk_id: {chunk_id}")
            continue

        reference_context_ids.append(chunk_id)
        relevant_chunks.append(description)

    golden_answer = generate_golden_answer(data.question, relevant_chunks)

    reference_dataset.append({
        "question": data.question,
        "ground_truth": golden_answer,
        "reference_context_ids": reference_context_ids,
        "reference_descriptions": relevant_chunks,
    })

Working on question 1
Working on question 2
Working on question 3
Working on question 4
Working on question 5
Working on question 6
Working on question 7
Working on question 8
Working on question 9
Working on question 10
Working on question 11
Working on question 12
Working on question 13
Working on question 14
Working on question 15
Working on question 16
Working on question 17
Working on question 18
Working on question 19
Working on question 20
Working on question 21
Working on question 22
Working on question 23
Working on question 24
Working on question 25
Working on question 26
Working on question 27
Working on question 28
Working on question 29
Working on question 30


In [116]:
reference_dataset

[{'question': 'Do you have a 128GB flash drive that works with both USB-C phones and regular USB ports on my computer?',
  'ground_truth': 'Yes — the DISAIN 128GB Dual Drive USB flash drive looks like a fit for that use. It has both a reversible USB Type-C connector and a traditional USB connector, so it can transfer files between USB-C phones/tablets and computers with regular USB ports. It also notes USB Type-C devices need OTG support. If you want, I can also point out the SanDisk 128GB option, but its description here doesn’t say it has both connector types.',
  'reference_context_ids': ['B07PHXDR1R',
   'B08P1GD671',
   'B07NLYRTK8',
   'B017TJN22C'],
  'reference_descriptions': ['128GB USB Flash Drive,DISAIN Dual Drive USB 3.0 Type-C 3.1 Gen 1 Devices, Thumb Drive Memory Stick for Computers ( Red) Flash drive for USB Type-C devices, with a reversible USB Type-C connector and a traditional USB connector. Reliable and secure storage for your videos, music, photos, or other files. E

### Create LangSmith Dataset

In [117]:
ls_client = Client()

In [118]:
dataset_name = "rag-evaluation-dataset-extended"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset for 1000 items"
)

In [119]:
for item in reference_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["ground_truth"],
            "reference_context_ids": item["reference_context_ids"],
            "reference_descriptions": item["reference_descriptions"]
        }
    )